# lab.py environment/target test

Exercises the `lab` API end to end -- declare, bind, initiate a run, run jobs
by hand, verify -- against whatever `lab.target` resolves to. See `lab.py`'s
own module docstring for the full environment/target model; the short
version:

- `lab.environment` is a fact ("local" or "modal"), auto-detected, never
  chosen.
- `lab.target` is a choice ("local" or "modal"), defaulting to
  `environment` -- so by default this notebook builds everything against
  local, disposable storage (`config.LOCAL_STORAGE`, under `.scratch/`,
  gitignored) when run locally, and against the real volume when run in
  JupyterLab. Nothing here needs a Modal deployment to fully exercise the
  declare/bind/run-jobs-by-hand flow -- that only becomes true once you
  deliberately call `lab.init(target="modal")`.

Run this in JupyterLab (declares/builds/binds against the real volume) or
locally in VS Code (declares/builds/binds against `.scratch/storage`) --
same code, same result shape, either way. The one place environment
genuinely matters is the bonus section at the end, which needs a real Modal
deployment to reach.

## Setup

In [9]:
import lab
lab.init(target='modal')
print(f"environment: {lab.environment}")
print(f"target:      {lab.target}")
print(f"root():      {lab.root()}")

environment: local
target:      modal
root():      /storage


## 1. Declare shared artifacts

A small, fast tree: one short public-domain text, one tiny tokenizer, one
tokenized source. Same shape as `tokenizer_demo.ipynb`, but through `lab.*`
instead of `dag.resolve` directly, and against the current target rather
than a notebook-local throwaway root -- these are shared artifacts (no
run_id), so re-running this notebook reuses whatever's already built on that
target rather than redeclaring it.

In [10]:
from sources.artifact import Source
from tokenizers.bpe import TokenizedSource, Tokenizer

romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)

# vocab_size=256 -- distinct from declare.ipynb (350), init.ipynb (1420) and
# tokenizer_demo.ipynb (1000), so this test's own artifacts are easy to spot
# and don't collide with a real run's tokenizer.
tokenizer = Tokenizer(
    vocab_size=256,
    special_tokens=("<pad>", "<unk>"),
    sources=(romeojuliet,),
)
tokenized = TokenizedSource(tokenizer=tokenizer, source=romeojuliet)

print(f"tokenizer path: {tokenizer.artifact_path}")
print(f"tokenized path: {tokenized.artifact_path}")

tokenizer path: tokenizers/bpe-256-abfcacd6b1
tokenized path: tokenizers/bpe-256-abfcacd6b1/bin/romeojuliet


In [11]:
# Preview: reconciles the whole tree against the current target, writes
# nothing. When target == environment (the default) this returns the
# Declaration itself (.rows/.problems/.ok); the one cross case (local
# environment targeting modal) returns None instead -- its report is
# printed there, same as main.declare's always was. See lab.declare's
# own docstring.
report = lab.declare(tokenized)  # commit defaults to False -- resolve + check only
if report is not None:
    print(report)

run - under /storage
  done       sources/romeojuliet
  done       tokenizers/bpe-256-abfcacd6b1
  done       tokenizers/bpe-256-abfcacd6b1/bin/romeojuliet

3 done
ok -- 0 to declare


In [5]:
# Now actually declare: writes manifests for anything still `new`.
result = lab.declare(tokenized, commit=True)
if result is not None:
    print(result)

run - under /Users/oguz/Projects/launchpad/.scratch/storage
  declared   sources/romeojuliet
  declared   tokenizers/bpe-256-abfcacd6b1
  declared   tokenizers/bpe-256-abfcacd6b1/bin/romeojuliet

3 declared
ok -- 0 to declare


## 2. Bind an artifact back, inspect its manifest

`lab.bind(path)` by path -- only ever allowed when `target == environment`
(see the permission matrix in `lab.py`'s module docstring), so this is
always a direct local read against whatever `lab.root()` currently is. Given
an artifact object instead of a path, `lab.bind` calls that object's own
`.bind(root)` directly (see part 5, and `lab.py`'s own docstring).

In [6]:
bound = lab.bind(str(tokenizer.artifact_path))

print(bound)                 # every parameter, read back off the manifest
print(bound.commit)          # the code it was built under
print(bound.deps())          # the artifacts underneath it (the source, here)
print(f"bound: {bound.bound}")

Tokenizer(vocab_size=256, special_tokens=('<pad>', '<unk>'), sources=(Source(name='romeojuliet', url='https://www.gutenberg.org/cache/epub/1513/pg1513.txt'),))
d79d373a116be7aefde16808b255c7434a41ef66-dirty
[Source(name='romeojuliet', url='https://www.gutenberg.org/cache/epub/1513/pg1513.txt')]
bound: False


## 3. Initiate a run

Same shape as `declare.ipynb`/`init.ipynb`: a fresh `run_id`, a dataset, a
pretraining config -- except the mock model family (no GPU needed, no real
training, just a fast simulated loss curve) and no `allocated_resources`,
since this is a cheap correctness test, not a real run. `RUN_ID` is
timestamp-free on purpose so re-running this cell doesn't pile up runs --
change it if you want a fresh one.

In [12]:
from mappeddatasets.artifact import MappedDataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig

RUN_ID = "RUN_LAB"  # fixed -- reruns reuse this run, don't pile up new ones

mapped = MappedDataSet.from_sources(
    tokenizer=tokenizer, train_sources=[romeojuliet], valid_sources=[romeojuliet]
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=mapped,
    tokenizer=tokenizer,
    model_parameters=ModelParameters(hidden_size=16, num_layers=1),
    config=PretrainingConfig(
        total_steps=50, batch_size=8, lr=1e-3, seed=1, checkpoint_every=25
    ),
)

report = lab.declare(pretraining)  # resolve + check only
if report is not None:
    print(report)

run RUN_LAB under /storage
  done       sources/romeojuliet
  done       tokenizers/bpe-256-abfcacd6b1
  done       tokenizers/bpe-256-abfcacd6b1/bin/romeojuliet
  done       mappeddatasets/mapped-6077a1fa8e
  new        runs/RUN_LAB/pretraining

4 done, 1 new
ok -- 1 to declare


In [13]:
result = lab.declare(pretraining, commit=True)
if result is not None:
    print(result)

run RUN_LAB under /storage
  done       sources/romeojuliet
  done       tokenizers/bpe-256-abfcacd6b1
  done       tokenizers/bpe-256-abfcacd6b1/bin/romeojuliet
  done       mappeddatasets/mapped-6077a1fa8e
  declared   runs/RUN_LAB/pretraining

1 declared, 4 done
ok -- 0 to declare


## 4. Run jobs manually

Runs every job the plan needs, in dependency order, using `lab.worker` --
the fake worker `lab.py` builds for exactly this: running a job by hand from
a cell, outside the run context (a real `run_job` call) it would otherwise
run under. No lease, no heartbeat, nothing to supersede it -- see
`lab.worker`'s own docstring in `lab.py`.

Works against whatever `lab.root()` currently resolves to -- `.scratch/storage`
locally, the volume in JupyterLab. No environment check needed here anymore:
this is exactly the "local testing runs jobs sequentially against local
storage" pattern the spec describes, and it's just as true when "local" means
the real volume from inside a container.

In [ ]:
import dag.resolve as dag_resolve

for to_build in (tokenized, pretraining):
    for job in dag_resolve.job_list(dag_resolve.resolve(to_build)):
        if dag_resolve.status(job.artifact, lab.root()) == "done":
            print(f"already done: {job.artifact.artifact_path}")
            continue
        print(f"running {type(job).__name__} for {job.artifact.artifact_path}")
        job.run(lab.root(), lab.worker)
lab.publish()  # land what these jobs just wrote -- no-op unless target is the volume

## 5. Verify, bind, use

`lab.refresh()` first, in case something else built these since this session
started -- no-op unless the target is the volume. Then re-check status, bind,
and actually use both artifacts: encode text with the tokenizer, read back
the run's progress file.

In [ ]:
lab.refresh()

report = lab.declare(tokenized)  # commit defaults to False -- resolve + check only
if report is not None:
    print(report)

report = lab.declare(pretraining)  # resolve + check only
if report is not None:
    print(report)

In [ ]:
bound_tokenizer = lab.bind(tokenizer)
print(f"{len(bound_tokenizer.vocab)} vocab entries")

text = "But soft, what light through yonder window breaks?"
ids = bound_tokenizer.encode(text)
print(ids)
print(repr(bound_tokenizer.decode(ids)))

progress_path = pretraining.paths(lab.root())["progress"]
if progress_path.exists():
    print(progress_path.read_text())
else:
    print("no progress file yet -- run part 4 first")

## 6. Publish

Land anything this session wrote (part 4 already published once, but this is
idempotent and harmless to call again -- a no-op unless the target is the
volume, same as `refresh`).

In [ ]:
lab.publish()

## Bonus: the cross-environment case

The one case `target == environment` doesn't cover: declaring against the
real volume from a local environment. Needs an actual Modal deployment to
fully reach (this cell's `declare` call will raise if there isn't one handy
-- that's an infra gap, not a `lab.py` bug). `commit` stays False (the
default) throughout, so this never writes anything -- just previews and
confirms the permission boundary.

In [ ]:
if lab.environment == "local":
    lab.init(target="modal")
    try:
        report = lab.declare(tokenized)  # allowed: local env, modal target
        print("declare against the real volume from here: OK")
        if report is not None:
            print(report)
    except Exception as exc:
        print(f"declare reached the permission check fine, failed after that (needs a real Modal deployment): {exc}")

    try:
        lab.bind(str(tokenizer.artifact_path))  # prohibited: local env, modal target
        print("FAIL: bind should have raised PermissionError")
    except PermissionError as exc:
        print(f"bind correctly refused: {exc}")
    finally:
        lab.init()  # reset to the default
else:
    print("skipping -- already in a modal environment, target is already 'modal', "
          "nothing to demonstrate here that section 1-6 didn't already cover")